# Notebook 01c: EIA-930 Historical BA-level Backfill (2019–2024)

**One Sensor, One Year — Edition 2: US Grid**

Stage A backfill for the data-center anchor. Notebook 11's three signatures came back 3/3 on 2025, so we're now pulling **per-BA** hourly fuel-mix for the same 10 balancing authorities back through 2019. With a 7-year per-BA panel we can ask whether the hyperscaler-vs-control gap has been widening as data-center capex ramped.

**Scope:** 10 BAs (`PJM, ERCO, MISO` hyperscaler + `BPAT, CISO, ISNE, NYIS, SOCO, SWPP, TVA` control) × 6 years (2019–2024) = 60 (BA, year) pulls. Each is independently cached on disk as `data/raw/eia930/<RESP>_<YEAR>.json` and reuses any cache already present from notebooks 01 / 01b.

**Outputs:** `data/processed/ba_{2019..2024}_hourly.parquet` (one per year, schema-matched to `ba_2025_hourly.parquet`), plus `data/processed/ba_2019_2025_hourly.parquet` (concat across all 7 years) for the trend notebook.

In [1]:
import json
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv

RAW = Path('../data/raw/eia930')
PROCESSED = Path('../data/processed')
RAW.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)

API_URL = 'https://api.eia.gov/v2/electricity/rto/fuel-type-data/data/'

# Same 10 BAs as notebook 11. US48 is excluded — already in 01b.
BAS = ['PJM', 'ERCO', 'MISO', 'BPAT', 'CISO', 'ISNE', 'NYIS', 'SOCO', 'SWPP', 'TVA']
YEARS = [2019, 2020, 2021, 2022, 2023, 2024]

load_dotenv(dotenv_path=Path('../../../.env'))
API_KEY = os.getenv('EIA_API_KEY')
assert API_KEY and len(API_KEY) == 32, 'EIA_API_KEY missing or wrong length in .env'
print(f'API key loaded (masked: {API_KEY[:4]}...{API_KEY[-4:]})')
print(f'Targets: {len(BAS)} BAs × {len(YEARS)} years = {len(BAS) * len(YEARS)} pulls')

API key loaded (masked: c09e...0e58)
Targets: 10 BAs × 6 years = 60 pulls


## 1. Fetch helper

Same shape as `fetch_eia930()` in notebook 01, parameterized by respondent and year. Pagination, exponential backoff, page-size halving on repeated 504s, per-pull JSON cache — all unchanged. Cache key is `<RESP>_<YEAR>.json` so any pre-existing 2025 BA caches and any US48 caches sit alongside without collision.

In [2]:
def fetch_eia930(respondent, year, sleep_s=0.25, page_size=5000, max_retries=4):
    """Pull all hourly fuel-mix rows for one (respondent, year). Cached per pull."""
    cache = RAW / f'{respondent}_{year}.json'
    if cache.exists():
        with open(cache) as f:
            data = json.load(f)
        print(f'  [{respondent} {year}] cached: {len(data):,} rows')
        return data

    base_params = {
        'api_key': API_KEY,
        'frequency': 'hourly',
        'data[0]': 'value',
        'facets[respondent][]': respondent,
        'start': f'{year}-01-01T00',
        'end': f'{year}-12-31T23',
        'sort[0][column]': 'period',
        'sort[0][direction]': 'asc',
    }

    all_rows, offset, current_page_size = [], 0, page_size
    while True:
        params = {**base_params, 'length': current_page_size, 'offset': offset}
        page = None
        for attempt in range(1, max_retries + 1):
            try:
                r = requests.get(API_URL, params=params, timeout=120)
                if r.status_code >= 500:
                    raise requests.HTTPError(f'{r.status_code} server error')
                r.raise_for_status()
                page = r.json().get('response', {}).get('data', [])
                break
            except (requests.HTTPError, requests.Timeout, requests.ConnectionError) as e:
                if attempt == max_retries:
                    raise RuntimeError(
                        f'  [{respondent} {year}] failed at offset={offset} after {max_retries} retries: {e}'
                    ) from None
                backoff = 2 ** attempt
                print(f'  [{respondent} {year}] offset={offset} attempt {attempt} failed ({e}); retrying in {backoff}s')
                time.sleep(backoff)
                if '504' in str(e) and current_page_size > 500:
                    current_page_size = max(500, current_page_size // 2)
                    params['length'] = current_page_size
                    print(f'  [{respondent} {year}] reduced page size to {current_page_size}')
        if not page:
            break
        all_rows.extend(page)
        offset += current_page_size
        if len(page) < current_page_size:
            break
        time.sleep(sleep_s)

    with open(cache, 'w') as f:
        json.dump(all_rows, f)
    print(f'  [{respondent} {year}] fetched: {len(all_rows):,} rows -> {cache.name}')
    return all_rows

## 2. Pull all (BA, year) combinations

60 pulls. With 0.25s rate-limit sleep and ~15 pages per BA-year, expect ~3–5 minutes wall clock on a cold cache. Re-runs read from disk and finish in seconds.

In [3]:
raw_by_pull = {}
t0 = time.time()
for year in YEARS:
    for ba in BAS:
        raw_by_pull[(ba, year)] = fetch_eia930(ba, year)
elapsed = time.time() - t0
total_rows = sum(len(v) for v in raw_by_pull.values())
print(f'\nDone: {total_rows:,} total rows across {len(raw_by_pull)} pulls in {elapsed:.0f}s')

  [PJM 2019] cached: 70,080 rows
  [ERCO 2019] cached: 61,320 rows


  [MISO 2019] cached: 61,320 rows
  [BPAT 2019] cached: 52,560 rows


  [CISO 2019] cached: 65,675 rows
  [ISNE 2019] cached: 70,080 rows


  [NYIS 2019] cached: 70,080 rows
  [SOCO 2019] cached: 70,072 rows


  [SWPP 2019] cached: 70,080 rows
  [TVA 2019] cached: 70,077 rows


...

  [ISNE 2024] fetched: 73,198 rows -> ISNE_2024.json


  [NYIS 2024] fetched: 70,246 rows -> NYIS_2024.json


  [SOCO 2024] fetched: 68,952 rows -> SOCO_2024.json


  [SWPP 2024] fetched: 70,264 rows -> SWPP_2024.json


  [TVA 2024] fetched: 70,195 rows -> TVA_2024.json

Done: 3,987,633 total rows across 60 pulls in 2587s


## 3. Build per-year long parquets

Schema target: `(period, respondent, fueltype, value_mwh)` — same as `ba_2025_hourly.parquet`. The parquet stays in long form so notebook 12 can load any year-slice without re-pivoting.

In [4]:
results = {}
for year in YEARS:
    frames = []
    for ba in BAS:
        rows = raw_by_pull[(ba, year)]
        if not rows:
            print(f'WARN: ({ba}, {year}) returned no data; skipping')
            continue
        df = pd.DataFrame(rows)
        df['period'] = pd.to_datetime(df['period'], format='%Y-%m-%dT%H')
        df['value_mwh'] = pd.to_numeric(df['value'], errors='coerce')
        df = df[['period', 'respondent', 'fueltype', 'value_mwh']]
        frames.append(df)

    if not frames:
        print(f'WARN: no frames for {year}')
        continue
    long = pd.concat(frames, ignore_index=True)
    out = PROCESSED / f'ba_{year}_hourly.parquet'
    long.to_parquet(out, index=False)

    results[year] = {
        'rows': len(long),
        'bas': long['respondent'].nunique(),
        'fuels': long['fueltype'].nunique(),
        'total_twh': long['value_mwh'].sum() / 1e6,
        'path': str(out),
    }
    print(f'  {year}: {len(long):,} rows  ({long["respondent"].nunique()} BAs × {long["fueltype"].nunique()} fuels)  '
          f'{long["value_mwh"].sum() / 1e6:,.1f} TWh  -> {out.name}')

  2019: 661,344 rows  (10 BAs × 8 fuels)  2,952.1 TWh  -> ba_2019_hourly.parquet


  2020: 656,198 rows  (10 BAs × 8 fuels)  2,843.2 TWh  -> ba_2020_hourly.parquet


  2021: 665,781 rows  (10 BAs × 8 fuels)  2,974.7 TWh  -> ba_2021_hourly.parquet


  2022: 665,685 rows  (10 BAs × 8 fuels)  3,091.8 TWh  -> ba_2022_hourly.parquet


  2023: 665,706 rows  (10 BAs × 8 fuels)  3,050.8 TWh  -> ba_2023_hourly.parquet


  2024: 672,919 rows  (10 BAs × 13 fuels)  3,148.8 TWh  -> ba_2024_hourly.parquet


## 4. Schema + coverage validation

Three checks:
1. Each new parquet matches `ba_2025_hourly.parquet`'s dtype signature exactly.
2. Row counts within ±5% of 2025 (allowing for fewer fuels reported in earlier years).
3. All 10 BAs present in every year.

In [5]:
ref = pd.read_parquet(PROCESSED / 'ba_2025_hourly.parquet')
ref_dtypes = ref.dtypes.to_dict()
ref_rows = len(ref)
ref_bas = sorted(ref['respondent'].unique())
print(f'Reference (2025): {ref_rows:,} rows, {len(ref_bas)} BAs, {ref["fueltype"].nunique()} fuels')
print(f'Reference dtypes: {ref_dtypes}\n')

audit = []
for year in YEARS:
    p = PROCESSED / f'ba_{year}_hourly.parquet'
    if not p.exists():
        audit.append({'year': year, 'status': 'MISSING'})
        continue
    df = pd.read_parquet(p)
    bas = sorted(df['respondent'].unique())
    dtypes_ok = df.dtypes.to_dict() == ref_dtypes
    bas_ok = bas == sorted(BAS)
    row_ratio = len(df) / ref_rows
    audit.append({
        'year': year,
        'rows': len(df),
        'row_ratio': round(row_ratio, 3),
        'bas': len(bas),
        'fuels': df['fueltype'].nunique(),
        'twh': round(df['value_mwh'].sum() / 1e6, 1),
        'dtypes_ok': dtypes_ok,
        'bas_ok': bas_ok,
    })

audit_df = pd.DataFrame(audit)
print(audit_df.to_string(index=False))

Reference (2025): 808,277 rows, 10 BAs, 16 fuels
Reference dtypes: {'period': dtype('<M8[us]'), 'respondent': <StringDtype(na_value=nan)>, 'fueltype': <StringDtype(na_value=nan)>, 'value_mwh': dtype('int64')}



 year   rows  row_ratio  bas  fuels    twh  dtypes_ok  bas_ok
 2019 661344      0.818   10      8 2952.1      False    True
 2020 656198      0.812   10      8 2843.2      False    True
 2021 665781      0.824   10      8 2974.7      False    True
 2022 665685      0.824   10      8 3091.8      False    True
 2023 665706      0.824   10      8 3050.8      False    True
 2024 672919      0.833   10     13 3148.8      False    True


## 5. Combined 2019–2025 parquet

Concat all 7 years (the 6 historical + the existing 2025) into a single long parquet for notebook 12. Adds a `year` column for fast filtering. Per-year files stay on disk for re-use elsewhere.

In [6]:
ALL_YEARS = sorted(YEARS + [2025])
frames = []
for year in ALL_YEARS:
    p = PROCESSED / f'ba_{year}_hourly.parquet'
    if not p.exists():
        print(f'WARN: missing {p.name}; skipping')
        continue
    df = pd.read_parquet(p)
    df['year'] = year
    frames.append(df)

combined = pd.concat(frames, ignore_index=True)
out = PROCESSED / 'ba_2019_2025_hourly.parquet'
combined.to_parquet(out, index=False)
print(f'Wrote {out.name} ({out.stat().st_size / 1e6:.1f} MB, {len(combined):,} rows)')
print(f'Year coverage: {sorted(combined["year"].unique())}')
print(f'BA coverage:   {sorted(combined["respondent"].unique())}')

# Annual TWh trajectory (sanity)
yearly = combined.groupby('year')['value_mwh'].sum() / 1e6
print('\nTotal generation across the 10 BAs (TWh):')
print(yearly.round(1).to_string())

Wrote ba_2019_2025_hourly.parquet (16.6 MB, 4,795,910 rows)
Year coverage: [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
BA coverage:   ['BPAT', 'CISO', 'ERCO', 'ISNE', 'MISO', 'NYIS', 'PJM', 'SOCO', 'SWPP', 'TVA']



Total generation across the 10 BAs (TWh):
year
2019    2952.1
2020    2843.2
2021    2974.7
2022    3091.8
2023    3050.8
2024    3148.8
2025    3244.0


## Next

Combined parquet ready for `12_data_center_signature_trend.ipynb` — reapply NB 11's three signature functions year-by-year and check whether the hyperscaler-vs-control gap is widening.